In [0]:
# Cell 1 - Create synthetic healthcare claims data
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import pyspark.sql.functions as F
from datetime import date

# Spark is pre-initialized in Databricks as 'spark'
data = [
    ("CLM001", "PAT001", "PRV001", "Z00.00", "99213", date(2024, 1, 5),  1200.00, "PAID",    "MEDICARE",   "NJ"),
    ("CLM002", "PAT002", "PRV002", "I10",    "93000", date(2024, 1, 12), 850.00,  "DENIED",  "MEDICAID",   "NY"),
    ("CLM003", "PAT003", "PRV001", "E11.9",  "83036", date(2024, 2, 3),  340.00,  "PAID",    "COMMERCIAL", "CA"),
    ("CLM004", "PAT004", "PRV003", "J45.901","94640", date(2024, 2, 18), 2100.00, "DENIED",  "MEDICARE",   "TX"),
    ("CLM005", "PAT005", "PRV002", "M54.5",  "72148", date(2024, 3, 7),  1750.00, "PAID",    "COMMERCIAL", "FL"),
    ("CLM006", "PAT006", "PRV004", "Z00.00", "99214", date(2024, 3, 22), 980.00,  "PENDING", "MEDICAID",   "NJ"),
    ("CLM007", "PAT007", "PRV001", "I25.10", "93454", date(2024, 4, 9),  4200.00, "PAID",    "MEDICARE",   "NY"),
    ("CLM008", "PAT008", "PRV003", "E11.9",  "83036", date(2024, 4, 14), 320.00,  "DENIED",  "COMMERCIAL", "CA"),
    ("CLM009", "PAT009", "PRV004", "J18.9",  "71046", date(2024, 5, 2),  1100.00, "PAID",    "MEDICARE",   "TX"),
    ("CLM010", "PAT010", "PRV002", "M79.3",  "20610", date(2024, 5, 28), 670.00,  "PAID",    "MEDICAID",   "FL"),
    ("CLM011", "PAT001", "PRV001", "Z00.00", "99213", date(2024, 6, 11), 1250.00, "DENIED",  "MEDICARE",   "NJ"),
    ("CLM012", "PAT003", "PRV003", "E11.9",  "83036", date(2024, 6, 19), 340.00,  "PAID",    "COMMERCIAL", "CA"),
    ("CLM013", "PAT005", "PRV002", "I10",    "93000", date(2024, 7, 4),  900.00,  "PAID",    "COMMERCIAL", "FL"),
    ("CLM014", "PAT007", "PRV004", "J45.901","94640", date(2024, 7, 23), 2200.00, "DENIED",  "MEDICARE",   "NY"),
    ("CLM015", "PAT009", "PRV001", "M54.5",  "72148", date(2024, 8, 8),  1800.00, "PAID",    "MEDICAID",   "TX"),
]

schema = StructType([
    StructField("claim_id",       StringType(),  False),
    StructField("patient_id",     StringType(),  False),
    StructField("provider_id",    StringType(),  False),
    StructField("diagnosis_code", StringType(),  True),
    StructField("procedure_code", StringType(),  True),
    StructField("claim_date",     DateType(),    False),
    StructField("paid_amount",    DoubleType(),  True),
    StructField("claim_status",   StringType(),  False),
    StructField("payer_type",     StringType(),  False),
    StructField("state",          StringType(),  False),
])

df = spark.createDataFrame(data, schema)
df.printSchema()
print(f"Total records: {df.count()}")
df.show(5)

root
 |-- claim_id: string (nullable = false)
 |-- patient_id: string (nullable = false)
 |-- provider_id: string (nullable = false)
 |-- diagnosis_code: string (nullable = true)
 |-- procedure_code: string (nullable = true)
 |-- claim_date: date (nullable = false)
 |-- paid_amount: double (nullable = true)
 |-- claim_status: string (nullable = false)
 |-- payer_type: string (nullable = false)
 |-- state: string (nullable = false)

Total records: 15
+--------+----------+-----------+--------------+--------------+----------+-----------+------------+----------+-----+
|claim_id|patient_id|provider_id|diagnosis_code|procedure_code|claim_date|paid_amount|claim_status|payer_type|state|
+--------+----------+-----------+--------------+--------------+----------+-----------+------------+----------+-----+
|  CLM001|    PAT001|     PRV001|        Z00.00|         99213|2024-01-05|     1200.0|        PAID|  MEDICARE|   NJ|
|  CLM002|    PAT002|     PRV002|           I10|         93000|2024-01-12|    

In [0]:
# Cell 2 - Data cleaning and feature engineering
from pyspark.sql.functions import (
    col, upper, trim, date_trunc, month, year,
    when, lit, datediff, current_date
)

# Clean and transform
df_clean = df \
    .withColumn("claim_status", upper(trim(col("claim_status")))) \
    .withColumn("payer_type",   upper(trim(col("payer_type")))) \
    .withColumn("state",        upper(trim(col("state")))) \
    .withColumn("claim_month",  date_trunc("month", col("claim_date"))) \
    .withColumn("claim_year",   year(col("claim_date"))) \
    .withColumn("is_denied",    when(col("claim_status") == "DENIED", 1).otherwise(0)) \
    .withColumn("is_paid",      when(col("claim_status") == "PAID", 1).otherwise(0)) \
    .withColumn("amount_bucket",
        when(col("paid_amount") < 500,  lit("LOW"))
       .when(col("paid_amount") < 1500, lit("MEDIUM"))
       .otherwise(lit("HIGH"))
    )

print("Cleaned DataFrame:")
df_clean.select("claim_id", "claim_status", "payer_type", "claim_month", "is_denied", "amount_bucket").show(10)

Cleaned DataFrame:
+--------+------------+----------+-------------------+---------+-------------+
|claim_id|claim_status|payer_type|        claim_month|is_denied|amount_bucket|
+--------+------------+----------+-------------------+---------+-------------+
|  CLM001|        PAID|  MEDICARE|2024-01-01 00:00:00|        0|       MEDIUM|
|  CLM002|      DENIED|  MEDICAID|2024-01-01 00:00:00|        1|       MEDIUM|
|  CLM003|        PAID|COMMERCIAL|2024-02-01 00:00:00|        0|          LOW|
|  CLM004|      DENIED|  MEDICARE|2024-02-01 00:00:00|        1|         HIGH|
|  CLM005|        PAID|COMMERCIAL|2024-03-01 00:00:00|        0|         HIGH|
|  CLM006|     PENDING|  MEDICAID|2024-03-01 00:00:00|        0|       MEDIUM|
|  CLM007|        PAID|  MEDICARE|2024-04-01 00:00:00|        0|         HIGH|
|  CLM008|      DENIED|COMMERCIAL|2024-04-01 00:00:00|        1|          LOW|
|  CLM009|        PAID|  MEDICARE|2024-05-01 00:00:00|        0|       MEDIUM|
|  CLM010|        PAID|  MEDICAID

In [0]:
# Cell 3 - Analytics: Payer-level denial rate and claims summary
from pyspark.sql.functions import count, sum, avg, round, col

# Payer-level summary
payer_summary = df_clean.groupBy("payer_type") \
    .agg(
        count("claim_id")                          .alias("total_claims"),
        sum("is_denied")                           .alias("denied_claims"),
        sum("is_paid")                             .alias("paid_claims"),
        round(avg("paid_amount"), 2)               .alias("avg_claim_amount"),
        round(sum("paid_amount"), 2)               .alias("total_paid_amount"),
        round(sum("is_denied") * 100.0 / count("claim_id"), 2).alias("denial_rate_pct")
    ) \
    .orderBy("denial_rate_pct", ascending=False)

print("=== PAYER-LEVEL DENIAL RATE ANALYSIS ===")
payer_summary.show()

# Monthly trend
monthly_trend = df_clean.groupBy("claim_month", "payer_type") \
    .agg(
        count("claim_id")                          .alias("total_claims"),
        round(sum("paid_amount"), 2)               .alias("total_amount"),
        round(sum("is_denied") * 100.0 / count("claim_id"), 2).alias("denial_rate_pct")
    ) \
    .orderBy("claim_month", "payer_type")

print("=== MONTHLY CLAIMS TREND BY PAYER ===")
monthly_trend.show(20)

=== PAYER-LEVEL DENIAL RATE ANALYSIS ===
+----------+------------+-------------+-----------+----------------+-----------------+---------------+
|payer_type|total_claims|denied_claims|paid_claims|avg_claim_amount|total_paid_amount|denial_rate_pct|
+----------+------------+-------------+-----------+----------------+-----------------+---------------+
|  MEDICARE|           6|            3|          3|         2008.33|          12050.0|           50.0|
|  MEDICAID|           4|            1|          2|          1075.0|           4300.0|           25.0|
|COMMERCIAL|           5|            1|          4|           730.0|           3650.0|           20.0|
+----------+------------+-------------+-----------+----------------+-----------------+---------------+

=== MONTHLY CLAIMS TREND BY PAYER ===
+-------------------+----------+------------+------------+---------------+
|        claim_month|payer_type|total_claims|total_amount|denial_rate_pct|
+-------------------+----------+------------+----

In [0]:
# Cell 4 - Window functions: running totals and rankings
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, sum, avg, round, col, dense_rank

# Window: rank providers by total claims
provider_window = Window.orderBy(col("total_claims").desc())

provider_summary = df_clean.groupBy("provider_id") \
    .agg(
        count("claim_id")                          .alias("total_claims"),
        round(sum("paid_amount"), 2)               .alias("total_amount"),
        round(sum("is_denied") * 100.0 / count("claim_id"), 2).alias("denial_rate_pct")
    ) \
    .withColumn("provider_rank", dense_rank().over(provider_window))

print("=== PROVIDER RANKING BY TOTAL CLAIMS ===")
provider_summary.orderBy("provider_rank").show()

# Window: running total of paid amount by payer over time
running_window = Window.partitionBy("payer_type").orderBy("claim_month") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

running_totals = df_clean.groupBy("payer_type", "claim_month") \
    .agg(round(sum("paid_amount"), 2).alias("monthly_amount")) \
    .withColumn("running_total", round(sum("monthly_amount").over(running_window), 2)) \
    .orderBy("payer_type", "claim_month")

print("=== RUNNING TOTAL PAID AMOUNT BY PAYER ===")
running_totals.show(20)

=== PROVIDER RANKING BY TOTAL CLAIMS ===


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+------------+------------+---------------+-------------+
|provider_id|total_claims|total_amount|denial_rate_pct|provider_rank|
+-----------+------------+------------+---------------+-------------+
|     PRV001|           5|      8790.0|           20.0|            1|
|     PRV002|           4|      4170.0|           25.0|            2|
|     PRV003|           3|      2760.0|          66.67|            3|
|     PRV004|           3|      4280.0|          33.33|            3|
+-----------+------------+------------+---------------+-------------+

=== RUNNING TOTAL PAID AMOUNT BY PAYER ===
+----------+-------------------+--------------+-------------+
|payer_type|        claim_month|monthly_amount|running_total|
+----------+-------------------+--------------+-------------+
|COMMERCIAL|2024-02-01 00:00:00|         340.0|        340.0|
|COMMERCIAL|2024-03-01 00:00:00|        1750.0|       2090.0|
|COMMERCIAL|2024-04-01 00:00:00|         320.0|       2410.0|
|COMMERCIAL|2024-06-01 0

In [0]:
# Cell 5 - Spark SQL: register as temp view and query with SQL
df_clean.createOrReplaceTempView("healthcare_claims")

# Query 1: Denial rate by diagnosis code
diagnosis_analysis = spark.sql("""
    SELECT
        diagnosis_code,
        COUNT(claim_id)                                         AS total_claims,
        SUM(is_denied)                                          AS denied_claims,
        ROUND(AVG(paid_amount), 2)                              AS avg_paid_amount,
        ROUND(SUM(is_denied) * 100.0 / COUNT(claim_id), 2)     AS denial_rate_pct
    FROM healthcare_claims
    GROUP BY diagnosis_code
    ORDER BY denial_rate_pct DESC
""")

print("=== DENIAL RATE BY DIAGNOSIS CODE ===")
diagnosis_analysis.show()

# Query 2: High-value denied claims (potential revenue recovery)
revenue_recovery = spark.sql("""
    SELECT
        claim_id,
        patient_id,
        provider_id,
        diagnosis_code,
        paid_amount,
        payer_type,
        state,
        claim_date
    FROM healthcare_claims
    WHERE claim_status = 'DENIED'
    AND   paid_amount > 500
    ORDER BY paid_amount DESC
""")

print("=== HIGH-VALUE DENIED CLAIMS (Revenue Recovery Candidates) ===")
revenue_recovery.show()

=== DENIAL RATE BY DIAGNOSIS CODE ===
+--------------+------------+-------------+---------------+---------------+
|diagnosis_code|total_claims|denied_claims|avg_paid_amount|denial_rate_pct|
+--------------+------------+-------------+---------------+---------------+
|       J45.901|           2|            2|         2150.0|         100.00|
|           I10|           2|            1|          875.0|          50.00|
|        Z00.00|           3|            1|        1143.33|          33.33|
|         E11.9|           3|            1|         333.33|          33.33|
|         M54.5|           2|            0|         1775.0|           0.00|
|        I25.10|           1|            0|         4200.0|           0.00|
|         J18.9|           1|            0|         1100.0|           0.00|
|         M79.3|           1|            0|          670.0|           0.00|
+--------------+------------+-------------+---------------+---------------+

=== HIGH-VALUE DENIED CLAIMS (Revenue Recovery Ca

In [0]:
# Cell 6 - Write to Delta table (Databricks native format)
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_claims_delta")

# Read back and verify
df_delta = spark.sql("SELECT * FROM healthcare_claims_delta LIMIT 5")
print("=== DATA WRITTEN TO DELTA TABLE SUCCESSFULLY ===")
df_delta.show()

# Show table stats
spark.sql("""
    SELECT
        COUNT(*)        AS total_records,
        COUNT(DISTINCT payer_type)    AS payer_types,
        COUNT(DISTINCT diagnosis_code) AS diagnosis_codes,
        MIN(claim_date) AS earliest_claim,
        MAX(claim_date) AS latest_claim
    FROM healthcare_claims_delta
""").show()

=== DATA WRITTEN TO DELTA TABLE SUCCESSFULLY ===
+--------+----------+-----------+--------------+--------------+----------+-----------+------------+----------+-----+-------------------+----------+---------+-------+-------------+
|claim_id|patient_id|provider_id|diagnosis_code|procedure_code|claim_date|paid_amount|claim_status|payer_type|state|        claim_month|claim_year|is_denied|is_paid|amount_bucket|
+--------+----------+-----------+--------------+--------------+----------+-----------+------------+----------+-----+-------------------+----------+---------+-------+-------------+
|  CLM001|    PAT001|     PRV001|        Z00.00|         99213|2024-01-05|     1200.0|        PAID|  MEDICARE|   NJ|2024-01-01 00:00:00|      2024|        0|      1|       MEDIUM|
|  CLM002|    PAT002|     PRV002|           I10|         93000|2024-01-12|      850.0|      DENIED|  MEDICAID|   NY|2024-01-01 00:00:00|      2024|        1|      0|       MEDIUM|
|  CLM003|    PAT003|     PRV001|         E11.9|   

In [0]:
# Cell 7 - Summary: Pipeline documentation
print("""
=== HEALTHCARE CLAIMS PYSPARK PIPELINE ===

Pipeline Steps:
1. Data Ingestion     - Loaded 15 claims records with enforced schema
2. Data Cleaning      - Standardized status/payer fields, engineered features
3. Payer Analytics    - Denial rate analysis: Medicare 50%, Medicaid 25%, Commercial 20%
4. Monthly Trends     - Claims volume and denial rates by payer per month
5. Window Functions   - Provider rankings, running totals using Spark Window API
6. Spark SQL          - Diagnosis-level denial analysis, revenue recovery identification
7. Delta Lake         - Persisted cleaned data as Delta table for downstream consumption

Key Findings:
- J45.901 (Asthma): 100% denial rate — highest risk diagnosis
- Medicare: highest avg claim amount ($2,008) and highest denial rate (50%)
- PRV003: highest denial rate among providers (66.67%)
- 4 high-value denied claims identified as revenue recovery candidates ($6,400 total)
""")


=== HEALTHCARE CLAIMS PYSPARK PIPELINE ===

Pipeline Steps:
1. Data Ingestion     - Loaded 15 claims records with enforced schema
2. Data Cleaning      - Standardized status/payer fields, engineered features
3. Payer Analytics    - Denial rate analysis: Medicare 50%, Medicaid 25%, Commercial 20%
4. Monthly Trends     - Claims volume and denial rates by payer per month
5. Window Functions   - Provider rankings, running totals using Spark Window API
6. Spark SQL          - Diagnosis-level denial analysis, revenue recovery identification
7. Delta Lake         - Persisted cleaned data as Delta table for downstream consumption

Key Findings:
- J45.901 (Asthma): 100% denial rate — highest risk diagnosis
- Medicare: highest avg claim amount ($2,008) and highest denial rate (50%)
- PRV003: highest denial rate among providers (66.67%)
- 4 high-value denied claims identified as revenue recovery candidates ($6,400 total)

